In [5]:
import scanpy as sc
import kaleidocell
import pandas as pd

In [4]:
adata = sc.read_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/01_extGBmap/immune_cells_extended_gbmap.h5ad")
print(adata.var_names[:10])

Index(['ENSG00000269696', 'ENSG00000230393', 'ENSG00000165244',
       'ENSG00000173597', 'ENSG00000158022', 'ENSG00000213018',
       'ENSG00000167220', 'ENSG00000231154', 'ENSG00000258457',
       'ENSG00000126106'],
      dtype='object')


In [7]:
print(kaleidocell.files)

Bundled files available in kaleidocell

h.all.v2026.1.Hs.symbols.gmt  [✓]
  MSigDB Hallmark gene sets (v2026.1, Human).
  50 coherent gene sets representing well-defined biological states and processes, curated to reduce redundancy across MSigDB.
  Source: https://www.gsea-msigdb.org/gsea/msigdb/.

c5.go.bp.v2026.1.Hs.symbols.gmt  [✓]
  MSigDB C5 GO Biological Process gene sets (v2026.1, Human).
  Gene sets derived from Gene Ontology (GO) biological process terms.
  Source: https://www.gsea-msigdb.org/gsea/msigdb/.

c5.go.cc.v2026.1.Hs.symbols.gmt  [✓]
  MSigDB C5 GO Cellular Component gene sets (v2026.1, Human).
  Gene sets derived from Gene Ontology (GO) cellular component terms.
  Source: https://www.gsea-msigdb.org/gsea/msigdb/.

c5.go.mf.v2026.1.Hs.symbols.gmt  [✓]
  MSigDB C5 GO Molecular Function gene sets (v2026.1, Human).
  Gene sets derived from Gene Ontology (GO) molecular function terms.
  Source: https://www.gsea-msigdb.org/gsea/msigdb/.

c6.all.v2026.1.Hs.symbols.gmt  [✓]

In [9]:
import inspect
import os

# Find where kaleidocell is installed
kaleidocell_path = os.path.dirname(inspect.getfile(kaleidocell))
print(kaleidocell_path)

/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell


In [10]:
for f in os.listdir(kaleidocell_path):
    if "hgnc" in f.lower() or "translation" in f.lower():
        print(f)

In [11]:
for root, dirs, files in os.walk(kaleidocell_path):
    for f in files:
        if "hgnc" in f.lower() or "translation" in f.lower():
            print(os.path.join(root, f))
            

/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/files/hgnc_ensembl_translation.txt


In [13]:
translation = pd.read_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/files/hgnc_ensembl_translation.txt", sep="\t")
print(translation.head())
print(translation.columns.tolist())

       HGNC       Ensembl_ID
0      A1BG  ENSG00000121410
1  A1BG-AS1  ENSG00000268895
2      A1CF  ENSG00000148584
3     A1S9T              NaN
4       A2M  ENSG00000175899
['HGNC', 'Ensembl_ID']


In [14]:
# Create mapping dictionary Ensembl -> HGNC
ensembl_to_hgnc = translation.dropna().set_index("Ensembl_ID")["HGNC"].to_dict()


In [15]:
# Map current gene names
new_names = [ensembl_to_hgnc.get(gene, gene) for gene in adata.var_names]

In [16]:
# Check how many were converted
n_converted = sum(1 for g, n in zip(adata.var_names, new_names) if g != n)
print(f"Converted {n_converted} out of {len(adata.var_names)} genes")

Converted 23016 out of 25987 genes


In [17]:
# Assign new names
adata.var_names = new_names
print(adata.var_names[:10])

Index(['ZNF470-DT', 'ENSG00000230393', 'ZNF367', 'SULT1B1', 'TRIM63',
       'PABPN1P1', 'HDHD2', 'MORF4L2-AS1', 'AJUBA-DT', 'TMEM53'],
      dtype='object')


In [18]:
adata.write_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/01_extGBmap/immune_cells_extended_gbmap_hgnc.h5ad")